In [2]:
import sys
import os

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

# Imports
from src.data.ingestion import MarketDataIngestion
from src.data.spark_pipeline import get_spark_session, SparkTechnicalIndicators
from src.data.databricks_client import DatabricksClient

print("Imports successful!")

Imports successful!


In [3]:
# 1. Fetch OHLCV
fetcher = MarketDataIngestion()
pandas_df = fetcher.fetch_daily_ohlcv("AAPL")

# 2. Spark Transformations
spark = get_spark_session()
spark_df = spark.createDataFrame(pandas_df)

indicator_calc = SparkTechnicalIndicators(spark)
processed_df = indicator_calc.compute_indicators(spark_df)

# 3. Store Data
db_client = DatabricksClient()
db_client.write_dataset(processed_df, table_name="aapl_indicators")

print("Phase 1 Execution Complete!")

[2026-09-21 05:21:45] [INFO] [stonks_maker]: Fetching OHLCV for AAPL via yfinance...
[2026-09-21 05:21:52] [INFO] [stonks_maker]: Starting Spark indicator transformations...
[2026-09-21 05:21:53] [INFO] [stonks_maker]: Completed Spark indicator calculations.
[2026-09-21 05:21:53] [INFO] [stonks_maker]: Databricks credentials not configured. Saving locally to Parquet.
[2026-09-21 05:21:56] [INFO] [stonks_maker]: Successfully saved to data/processed/aapl_indicators
Phase 1 Execution Complete!


In [4]:
# Verify saved Parquet/Delta dataset
output_df = db_client.read_dataset(spark, table_name="aapl_indicators")
output_df.show(5)

[2026-09-21 05:22:32] [INFO] [stonks_maker]: Reading dataset locally from data/processed/aapl_indicators
+----------+------+------------------+-----------------+------------------+------------------+---------+------------------+------------------+------------------+------------------+-----------------+
|      date|ticker|              open|             high|               low|             close|   volume|            sma_20|            sma_50|   bollinger_upper|   bollinger_lower|           rsi_14|
+----------+------+------------------+-----------------+------------------+------------------+---------+------------------+------------------+------------------+------------------+-----------------+
|2025-09-19|  AAPL|240.34217704154452|245.3935247988757|239.32594197772087|244.59646606445312|163741300|244.59646606445312|244.59646606445312|              NULL|              NULL|              0.0|
|2025-09-22|  AAPL|247.38614726921998|255.6954638689992|247.20680184487878| 255.1374969482422|10551